In [ ]:
# =============================================================================
# IMPORTACAO DE DADOS GPKG -> PostgreSQL  |  AULA 12+13
# Sobreposicao fundiaria, Desmatamento e Amazonia Legal
# =============================================================================
import geopandas as gpd
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path

# =============================================================================
# CONFIGURACOES - ALTERE CONFORME NECESSARIO
# =============================================================================

# Pasta onde estao os arquivos GPKG
PASTA_DADOS = Path(r"C:\Temp")

# =============================================================================
# ARQUIVOS 
# O script tenta automaticamente .gpkg e depois .shp
# =============================================================================
CAR_MT          = "es_mt_car_20260406"
UC              = "pa_br_ucs_mma_2026"
TI              = "pa_br_terrasindigenas_funai_2026"
TQ              = "pa_br_territoriosquilombolas_incra_2026"
CNFP            = "pa_br_cnfp_sfb_2024_retificado"
ASSENTAMENTOS   = "pa_br_assentamentos_incra_2026"
MALHA_MT        = "pa_mt_malhafundiaria_2025_cdt"
MODULOS_FISCAIS = "pa_br_modulosfiscais_incra"
SIGEF           = "pa_br_sigef_privado_incra_2026"
SNCI            = "pa_br_snci_privado_incra_2026"
DESMATAMENTO_MT = "pa_mt_desmatamento_inpe_20260304"
AMAZONIA_LEGAL  = "pa_br_amazonialegal_ibge_2022"
FAIXA_FRONTEIRA = "pa_br_faixafronteira_ibge_2024"

# =============================================================================
# CONTROLE DE IMPORTACAO
# True  = importa (cria/substitui a tabela)
# False = pula
# =============================================================================
IMPORTAR_CAR_MT          = True
IMPORTAR_UC              = True
IMPORTAR_TI              = True
IMPORTAR_TQ              = True
IMPORTAR_CNFP            = True
IMPORTAR_ASSENTAMENTOS   = True
IMPORTAR_MALHA_MT        = True
IMPORTAR_MODULOS_FISCAIS = True
IMPORTAR_SIGEF           = True
IMPORTAR_SNCI            = True
IMPORTAR_DESMATAMENTO_MT = True
IMPORTAR_AMAZONIA_LEGAL  = True
IMPORTAR_FAIXA_FRONTEIRA = True

# Se False, mantem a tabela existente no banco (nao reimporta)
RECRIAR_TABELAS = False

# =============================================================================
# CREDENCIAIS DO BANCO LOCAL (geotec)
# =============================================================================
DB_HOST     = "localhost"
DB_PORT     = "5432"
DB_USER     = "postgres"
DB_PASSWORD = "postgres"
DB_NAME     = "geotec"

# =============================================================================
# MAPEAMENTO: arquivo GPKG/SHP -> (schema, tabela)
# =============================================================================
IMPORTACOES = [
    (CAR_MT,          "car",    "es_mt_car_20260406",                     IMPORTAR_CAR_MT),
    (UC,              "mma",    "pa_br_ucs_mma_2026",                     IMPORTAR_UC),
    (TI,              "funai",  "pa_br_terrasindigenas_funai_2026",        IMPORTAR_TI),
    (TQ,              "incra",  "pa_br_territoriosquilombolas_incra_2026", IMPORTAR_TQ),
    (CNFP,            "sfb",    "pa_br_cnfp_sfb_2024_retificado",          IMPORTAR_CNFP),
    (ASSENTAMENTOS,   "incra",  "pa_br_assentamentos_incra_2026",         IMPORTAR_ASSENTAMENTOS),
    (MALHA_MT,        "malha",  "pa_mt_malhafundiaria_2025_cdt",          IMPORTAR_MALHA_MT),
    (MODULOS_FISCAIS, "incra",  "pa_br_modulosfiscais_incra",             IMPORTAR_MODULOS_FISCAIS),
    (SIGEF,           "incra",  "pa_br_sigef_privado_incra_2026",         IMPORTAR_SIGEF),
    (SNCI,            "incra",  "pa_br_snci_privado_incra_2026",          IMPORTAR_SNCI),
    (DESMATAMENTO_MT, "inpe",   "pa_mt_desmatamento_inpe_20260304",       IMPORTAR_DESMATAMENTO_MT),
    (AMAZONIA_LEGAL,  "ibge",   "pa_br_amazonialegal_ibge_2022",           IMPORTAR_AMAZONIA_LEGAL),
    (FAIXA_FRONTEIRA, "ibge",   "pa_br_faixafronteira_ibge_2024",          IMPORTAR_FAIXA_FRONTEIRA),
]

# =============================================================================
# FUNCOES
# =============================================================================

def conectar_banco():
    return create_engine(
        f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    )

def encontrar_arquivo(pasta, nome_arquivo):
    nome_base = Path(nome_arquivo).stem
    
    # Tenta GPKG primeiro
    gpkg_path = pasta / f"{nome_base}.gpkg"
    if gpkg_path.exists():
        return gpkg_path, 'gpkg'
    
    # Tenta SHP
    shp_path = pasta / f"{nome_base}.shp"
    if shp_path.exists():
        return shp_path, 'shp'
    
    return None, None

def criar_schemas(engine):
    schemas = ["car", "mma", "funai", "incra", "sfb", "malha", "inpe", "ibge"]
    with engine.connect() as conn:
        for s in schemas:
            conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {s}"))
        conn.execute(text("CREATE EXTENSION IF NOT EXISTS postgis"))
        conn.commit()
    print(f"[OK] Schemas prontos: {', '.join(schemas)}")


def tabela_existe(engine, schema, tabela):
    try:
        with engine.connect() as conn:
            conn.execute(text(f"SELECT 1 FROM {schema}.{tabela} LIMIT 1"))
            return True
    except Exception:
        return False


def contar_registros(engine, schema, tabela):
    try:
        with engine.connect() as conn:
            r = conn.execute(text(f"SELECT COUNT(*) FROM {schema}.{tabela}"))
            return r.fetchone()[0]
    except Exception:
        return 0


def importar_arquivo(engine, caminho, tipo_arquivo, schema, tabela):
    """
    Importa arquivo (GPKG ou SHP) para o PostgreSQL
    """
    print(f"  Lendo: {caminho.name} (formato: {tipo_arquivo.upper()})")
    
    try:
        gdf = gpd.read_file(str(caminho))
    except Exception as e:
        print(f"  ERRO ao ler arquivo: {e}")
        return False
    
    print(f"  Registros: {len(gdf)}")

    tem_geometria = (
        isinstance(gdf, gpd.GeoDataFrame)
        and gdf.active_geometry_name is not None
        and not gdf.geometry.isna().all()
    )

    if tem_geometria:
        if gdf.crs is None:
            gdf = gdf.set_crs(epsg=4674)
        else:
            gdf = gdf.to_crs(epsg=4674)
        gdf.to_postgis(
            name=tabela, con=engine, schema=schema,
            if_exists="replace", index=False
        )
    else:
        df = pd.DataFrame(gdf)
        geom_col = getattr(gdf, "active_geometry_name", None)
        if geom_col:
            df = df.drop(columns=geom_col, errors="ignore")
        print("  [INFO] Sem geometria ativa; importando como tabela comum")
        df.to_sql(
            name=tabela, con=engine, schema=schema,
            if_exists="replace", index=False
        )

    print(f"  OK -> {schema}.{tabela}")
    return True


def importar_se_necessario(engine, nome_arquivo, schema, tabela, ativo):
    """
    Versão atualizada: tenta GPKG, depois SHP
    """
    if not ativo:
        print(f"  [SKIP] {schema}.{tabela} (desativado)")
        return

    caminho, tipo_arquivo = encontrar_arquivo(PASTA_DADOS, nome_arquivo)
    
    if caminho is None:
        print(f"  [ERRO] Arquivo nao encontrado: {nome_arquivo} (GPKG ou SHP) em {PASTA_DADOS}")
        return

    if tabela_existe(engine, schema, tabela) and not RECRIAR_TABELAS:
        n = contar_registros(engine, schema, tabela)
        print(f"  [EXISTE] {schema}.{tabela} ({n} registros) - mantido")
        return

    importar_arquivo(engine, caminho, tipo_arquivo, schema, tabela)


# =============================================================================
# MAIN
# =============================================================================

def main():
    print("=" * 65)
    print("IMPORTACAO DE DADOS - AULA 12+13")
    print("=" * 65)

    if not PASTA_DADOS.exists():
        print(f"[ERRO] Pasta nao encontrada: {PASTA_DADOS}")
        return

    engine = conectar_banco()
    print("[OK] Conectado ao PostgreSQL")

    criar_schemas(engine)

    for i, (arquivo, schema, tabela, ativo) in enumerate(IMPORTACOES, 1):
        print(f"\n[{i}/{len(IMPORTACOES)}] {schema}.{tabela}")
        importar_se_necessario(engine, arquivo, schema, tabela, ativo)

    print("\n" + "=" * 65)
    print("RESUMO DAS TABELAS")
    print("=" * 65)
    for arquivo, schema, tabela, ativo in IMPORTACOES:
        if ativo:
            n = contar_registros(engine, schema, tabela)
            print(f"  {schema}.{tabela}: {n} registros")

    engine.dispose()
    print("\n[OK] Importacao concluida!")
    print("=" * 65)


main()


IMPORTACAO DE DADOS - AULA 12+13
[OK] Conectado ao PostgreSQL
[OK] Schemas prontos: car, mma, funai, incra, sfb, malha, inpe, ibge

[1/13] car.es_mt_car_20260406
  [EXISTE] car.es_mt_car_20260406 (213912 registros) - mantido

[2/13] mma.pa_br_ucs_mma_2026
  Lendo: pa_br_ucs_mma_2026.gpkg (formato: GPKG)
  Registros: 157
  OK -> mma.pa_br_ucs_mma_2026

[3/13] funai.pa_br_terrasindigenas_funai_2026
  Lendo: pa_br_terrasindigenas_funai_2026.gpkg (formato: GPKG)
  Registros: 94
  OK -> funai.pa_br_terrasindigenas_funai_2026

[4/13] incra.pa_br_territoriosquilombolas_incra_2026
  Lendo: pa_br_territoriosquilombolas_incra_2026.gpkg (formato: GPKG)
  Registros: 7
  OK -> incra.pa_br_territoriosquilombolas_incra_2026

[5/13] sfb.pa_br_cnfp_sfb_2024_retificado
  Lendo: pa_br_cnfp_sfb_2024_retificado.gpkg (formato: GPKG)
  Registros: 2010
  OK -> sfb.pa_br_cnfp_sfb_2024_retificado

[6/13] incra.pa_br_assentamentos_incra_2026
  Lendo: pa_br_assentamentos_incra_2026.gpkg (formato: GPKG)
  Registro